In [4]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

# 1.加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

client = MultiServerMCPClient({
    "time-mcp": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "time-mcp"]
    },
    "12306-mcp": {
        "transport": "streamable_http",
        "url": "https://mcp.api-inference.modelscope.net/4a607618cada4c/mcp"
        # 到这里获取，24小时有效，https://modelscope.cn/mcp/servers/@Joooook/12306-mcp/
    }
})
tools = await client.get_tools()

# 4.创建智能体
agent = create_agent(model, tools)

# 5.调用模型，流式调用
response = await agent.ainvoke(
    {"messages": [HumanMessage("现在是几点")]},
)

print(response)

Failed to parse JSONRPC message from server
Traceback (most recent call last):
  File "/Users/xlh/Documents/learn/ai/python/ai_insurance/ai_insurance_backend/.venv/lib/python3.12/site-packages/mcp/client/stdio/__init__.py", line 155, in stdout_reader
    message = types.JSONRPCMessage.model_validate_json(line)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/xlh/Documents/learn/ai/python/ai_insurance/ai_insurance_backend/.venv/lib/python3.12/site-packages/pydantic/main.py", line 845, in model_validate_json
    return cls.__pydantic_validator__.validate_json(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for JSONRPCMessage
  Invalid JSON: expected value at line 1 column 1 [type=json_invalid, input_value='Starting Time MCP server...', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid
Failed to parse JSONRPC message from server
Traceback (most 

{'messages': [HumanMessage(content='现在是几点', additional_kwargs={}, response_metadata={}, id='b2995cf0-c9ad-4b75-ae59-ba0513b9786a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 3179, 'total_tokens': 3242, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 768, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 768, 'prompt_cache_miss_tokens': 2411}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'b4aba5b4-32b5-4655-9f47-68541e8c3b87', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19b57-553f-75bb-a8ae-94c7c2578f41-0', tool_calls=[{'name': 'current_time', 'args': {'format': 'YYYY-MM-DD HH:mm:ss', 'timezone': 'Asia/Shanghai'}, 'id': 'call_00_LcHjApjnXwqKXKED22Rm4280', 'type': 'tool_call'}], invalid_tool_call

In [ ]:
# 5.invoke调用Agent， 传入的Dict，里面有messages
response = await agent.ainvoke(
    {"messages": [HumanMessage("帮我查一下下周四从北京到杭州的高铁")]}
)
for message in response["messages"]:
    message.pretty_print()